# San Diego Rental Property Data

Analysis of rental-related datasets: business tax certificates, STRO licenses, TOT establishments, city properties, and city leases.

## 1. Load Datasets

In [1]:
import polars as pl
import folium
from folium.plugins import HeatMap
from pathlib import Path
import math

DATA_DIR = Path("data")
PERMITS_DIR = Path("../building_permits/data")

In [2]:
business_files = [
    "sd_businesses_active_datasd.csv",
    "sd_businesses_inactive_1990to2000_datasd.csv",
    "sd_businesses_inactive_2000to2010_datasd.csv",
    "sd_businesses_inactive_2010to2015_datasd.csv",
    "sd_businesses_inactive_2015tocurr_datasd.csv",
]

business_dfs = []
for f in business_files:
    df = pl.read_csv(
        DATA_DIR / f,
        schema_overrides={"address_no": pl.String},
        infer_schema_length=10000,
    )
    business_dfs.append(df)

business_certs = pl.concat(business_dfs, how="diagonal_relaxed")
print(f"Business tax certificates: {business_certs.shape[0]:,} rows, {business_certs.shape[1]} cols")

Business tax certificates: 517,983 rows, 27 cols


In [3]:
stro = pl.read_csv(DATA_DIR / "stro_licenses_datasd.csv", infer_schema_length=10000)
stro = stro.with_columns(
    pl.col("latitude").cast(pl.Float64, strict=False),
    pl.col("longitude").cast(pl.Float64, strict=False),
)
print(f"STRO licenses: {stro.shape[0]:,} rows, {stro.shape[1]} cols")

STRO licenses: 8,328 rows, 23 cols


In [4]:
tot = pl.read_csv(DATA_DIR / "tot_establishments_datasd.csv", infer_schema_length=10000)
print(f"TOT establishments: {tot.shape[0]:,} rows, {tot.shape[1]} cols")

TOT establishments: 15,050 rows, 8 cols


In [5]:
city_properties = pl.read_csv(
    DATA_DIR / "city_property_details_datasd.csv",
    infer_schema_length=10000,
    schema_overrides={
        "land_cost": pl.String,
        "building_cost": pl.String,
        "closing_cost": pl.String,
    },
)
print(f"City properties: {city_properties.shape[0]:,} rows, {city_properties.shape[1]} cols")

City properties: 4,091 rows, 21 cols


In [6]:
city_leases = pl.read_csv(
    DATA_DIR / "city_property_leases_datasd.csv",
    infer_schema_length=10000,
    schema_overrides={"cost_line_amt_USD": pl.String},
)
print(f"City leases: {city_leases.shape[0]:,} rows, {city_leases.shape[1]} cols")

City leases: 825 rows, 14 cols


## 2. Data Dictionaries

In [7]:
business_dict = pl.read_csv(DATA_DIR / "sd_businesses_dictionary_datasd.csv")
print("Business Tax Certificates Dictionary")
business_dict

Business Tax Certificates Dictionary


field,description,possible_values
str,str,str
"""account_key""","""Primary Key""",null
"""account_status""","""Whether the account is active …","""Active; Inactive"""
"""account_creation_dt""","""Date the account was created a…",null
"""cert_expiration_dt""","""Expiration of current Business…",null
"""business_owner_name""","""Business Owner""",null
…,…,…
"""state""","""Business State""",null
"""zip""","""Business Zip""",null
"""pmb_box""","""Private Mail Box - PO Box loca…",null


In [8]:
stro_dict = pl.read_csv(DATA_DIR / "stro_licenses_dictionary_datasd.csv")
print("STRO Licenses Dictionary")
stro_dict

STRO Licenses Dictionary


field,description,possible_values
str,str,str
"""license_id""","""Unique identifier for license""",null
"""address""","""Street address of dwelling uni…",null
"""tier""","""The tier of license""","""See tier definition file"""
"""community_planning_area""","""The Community Planning Area in…",null
"""date_expiration""","""The date on which the license …",null
"""rtax_no""","""Rental Unit Business Tax accou…",null
"""tot_no""","""Transient Occupancy Tax (TOT) …",null


In [9]:
stro_tiers = pl.read_csv(DATA_DIR / "stro_licenses_tier_definitions_datasd.csv")
print("STRO Tier Definitions")
stro_tiers

STRO Tier Definitions


Tier,Number of aggregate rental days per year required,Host must reside onsite at least 275 days per year,City of San Diego Community Planning Area
i64,str,str,str
1,"""20 or fewer""","""No""","""All"""
2,"""More than 20""","""Yes""","""All"""
3,"""90 or more""","""No""","""All excluding Mission Beach Pr…"
4,"""90 or more""","""No""","""Mission Beach Precise Planning…"


In [10]:
tot_dict = pl.read_csv(DATA_DIR / "tot_establishments_datasd_dict.csv")
print("TOT Establishments Dictionary")
tot_dict

TOT Establishments Dictionary


field,description,possible_values
str,str,str
"""certificate_no""","""Unique Transient Occupancy Tax…",null
"""status""","""Status of certificate""","""1: Active"""
"""certificate_type""","""Must be selected by each appli…","""7: Vacation"""
"""property_address""","""The address of the property wh…",null
"""property_city""","""The city of the property where…",null
"""property_state""","""The state of the property wher…",null
"""property_zip""","""The 5 digit ZIP code of the pr…",null
"""date_created""","""This is the date the certifica…","""Certificates created January 1…"


In [11]:
city_props_dict = pl.read_csv(DATA_DIR / "city_property_details_dictionary_datasd.csv")
print("City Property Details Dictionary")
city_props_dict

City Property Details Dictionary


field,description,possible_values
str,str,str
"""site_code""","""Unique identifier for a City-o…",null
"""File Code""","""Number for the file associated…",null
"""grantor""","""Entity or individual who grant…",null
"""month_acquired""","""Month the property was acquire…",null
"""year_acquired""","""Year the property was acquired""",null
…,…,…
"""dedicated_park""","""Whether the site is a dedicate…","""Y-Yes, N-No"""
"""water_use""","""Whether site is a Water (Publi…","""W-Water; M-Metro Wastewater; N…"
"""use_restrictions""","""Any restrictions placed on the…",null


In [12]:
city_leases_dict = pl.read_csv(DATA_DIR / "city_property_leases_dictionary_datasd.csv", encoding="utf8-lossy")
print("City Property Leases Dictionary")
city_leases_dict

City Property Leases Dictionary


Field,Description,Possible values
str,str,str
"""lessee_name""","""Name of lessee""",null
"""lessee_company""","""Company of lessee, if applicab…",null
"""lessee_DBA""","""The name under which the lesse…",null
"""lessee_ZIP""","""ZIP code of lessee""",null
"""lease_record_type""","""Type of lease. The type of lea…",null
…,…,…
"""nonprofit_lessee""","""Whether the lessee is a nonpro…","""1-nonprofit, Blank-not a nonpr…"
"""effective_date""","""Date the lease was effective""",null
"""sched_termination_date""","""Date the lease is scheduled fo…",null


## 3. Schema and Data Quality Audit

In [13]:
def data_quality_audit(df: pl.DataFrame, name: str) -> pl.DataFrame:
    """Produces a data quality summary for the given dataframe.

    Reports column dtype, null count/percentage, and empty string
    count/percentage for each column.
    """
    n = df.shape[0]
    records = []
    for col in df.columns:
        null_count = df[col].null_count()
        if df[col].dtype == pl.String:
            empty_count = df.filter(pl.col(col) == "").shape[0]
        else:
            empty_count = 0
        records.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": null_count,
            "null_pct": round(null_count / n * 100, 2) if n > 0 else 0.0,
            "empty_str_count": empty_count,
            "empty_str_pct": round(empty_count / n * 100, 2) if n > 0 else 0.0,
        })
    audit = pl.DataFrame(records)
    print(f"\n{'=' * 60}")
    print(f"Data Quality Audit: {name} ({n:,} rows)")
    print(f"{'=' * 60}")
    return audit

In [14]:
data_quality_audit(business_certs, "Business Tax Certificates")


Data Quality Audit: Business Tax Certificates (517,983 rows)


column,dtype,null_count,null_pct,empty_str_count,empty_str_pct
str,str,i64,f64,i64,f64
"""account_key""","""Float64""",0,0.0,0,0.0
"""account_status""","""String""",0,0.0,0,0.0
"""date_account_creation""","""String""",0,0.0,0,0.0
"""date_cert_expiration""","""String""",0,0.0,0,0.0
"""date_cert_effective""","""String""",0,0.0,16,0.0
…,…,…,…,…,…
"""address_po_box""","""String""",0,0.0,505009,97.5
"""bid""","""String""",0,0.0,453800,87.61
"""council_district""","""String""",0,0.0,144591,27.91


In [15]:
data_quality_audit(stro, "STRO Licenses")


Data Quality Audit: STRO Licenses (8,328 rows)


column,dtype,null_count,null_pct,empty_str_count,empty_str_pct
str,str,i64,f64,i64,f64
"""license_id""","""String""",0,0.0,0,0.0
"""address""","""String""",0,0.0,0,0.0
"""street_number""","""String""",0,0.0,1,0.01
"""street_number_fraction""","""String""",0,0.0,8255,99.12
"""street_direction""","""String""",0,0.0,8105,97.32
…,…,…,…,…,…
"""latitude""","""Float64""",2,0.02,0,0.0
"""local_contact_contact_name""","""String""",0,0.0,0,0.0
"""local_contact_phone""","""Int64""",0,0.0,0,0.0


In [16]:
data_quality_audit(tot, "TOT Establishments")


Data Quality Audit: TOT Establishments (15,050 rows)


column,dtype,null_count,null_pct,empty_str_count,empty_str_pct
str,str,i64,f64,i64,f64
"""certificate_no""","""Float64""",0,0.0,0,0.0
"""status""","""Float64""",0,0.0,0,0.0
"""certificate_type""","""Float64""",0,0.0,0,0.0
"""property_address""","""String""",0,0.0,0,0.0
"""property_city""","""String""",0,0.0,0,0.0
"""property_state""","""String""",0,0.0,2,0.01
"""property_zip""","""Int64""",0,0.0,0,0.0
"""date_created""","""String""",0,0.0,0,0.0


In [17]:
data_quality_audit(city_properties, "City Properties")


Data Quality Audit: City Properties (4,091 rows)


column,dtype,null_count,null_pct,empty_str_count,empty_str_pct
str,str,i64,f64,i64,f64
"""site_code""","""String""",0,0.0,0,0.0
"""file_code""","""String""",0,0.0,7,0.17
"""grantor""","""String""",0,0.0,74,1.81
"""month_acquired""","""String""",0,0.0,29,0.71
"""year_acquired""","""String""",0,0.0,8,0.2
…,…,…,…,…,…
"""dedicated_park""","""String""",0,0.0,6,0.15
"""water_use""","""String""",0,0.0,0,0.0
"""use_restrictions""","""String""",0,0.0,0,0.0


In [18]:
data_quality_audit(city_leases, "City Leases")


Data Quality Audit: City Leases (825 rows)


column,dtype,null_count,null_pct,empty_str_count,empty_str_pct
str,str,i64,f64,i64,f64
"""site_code""","""String""",0,0.0,0,0.0
"""lessee_name""","""String""",0,0.0,0,0.0
"""lessee_company""","""String""",0,0.0,88,10.67
"""lessee_DBA""","""String""",0,0.0,754,91.39
"""address_zip""","""String""",0,0.0,90,10.91
…,…,…,…,…,…
"""nonprofit_lessee""","""Float64""",0,0.0,0,0.0
"""date_effective""","""String""",0,0.0,8,0.97
"""date_sched_termination""","""String""",0,0.0,200,24.24


In [19]:
print("Date format samples")
print("\nBusiness certs date_account_creation:")
print(business_certs["date_account_creation"].drop_nulls().head(5).to_list())
print("\nSTRO date_expiration:")
print(stro["date_expiration"].drop_nulls().head(5).to_list())
print("\nTOT date_created:")
print(tot["date_created"].drop_nulls().head(5).to_list())
print("\nCity leases date_effective:")
print(city_leases["date_effective"].drop_nulls().head(5).to_list())

Date format samples

Business certs date_account_creation:
['1974-07-01 12:00:00', '1974-07-01 12:00:00', '1974-07-01 12:00:00', '1974-07-01 12:00:00', '1974-07-01 12:00:00']

STRO date_expiration:
['2027-04-30', '2027-04-30', '2027-04-30', '2027-04-30', '2027-04-30']

TOT date_created:
['2008-04-18', '2011-10-14', '2008-04-18', '2011-10-14', '2011-10-14']

City leases date_effective:
['4/18/2019', '12/1/2016', '3/12/1974', '11/4/2021', '4/8/1980']


## 4. Cleaning

Replace empty strings with null and parse date columns.

In [20]:
def empty_to_null(df: pl.DataFrame) -> pl.DataFrame:
    """Replaces empty strings with null across all String columns."""
    return df.with_columns(
        pl.col(col).replace("", None)
        for col in df.columns
        if df[col].dtype == pl.String
    )

business_certs = empty_to_null(business_certs)
stro = empty_to_null(stro)
tot = empty_to_null(tot)
city_properties = empty_to_null(city_properties)
city_leases = empty_to_null(city_leases)
print("Empty strings replaced with null in all datasets.")

Empty strings replaced with null in all datasets.


In [21]:
business_date_cols = [
    "date_account_creation",
    "date_cert_expiration",
    "date_cert_effective",
    "date_business_start",
]
for col in business_date_cols:
    if col in business_certs.columns and business_certs[col].dtype == pl.String:
        business_certs = business_certs.with_columns(
            pl.col(col).str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False).alias(col)
        )

if "date_expiration" in stro.columns and stro["date_expiration"].dtype == pl.String:
    stro = stro.with_columns(
        pl.col("date_expiration").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
    )

if "date_created" in tot.columns and tot["date_created"].dtype == pl.String:
    tot = tot.with_columns(
        pl.col("date_created").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
    )

lease_date_cols = ["date_effective", "date_sched_termination"]
for col in lease_date_cols:
    if col in city_leases.columns and city_leases[col].dtype == pl.String:
        city_leases = city_leases.with_columns(
            pl.col(col).str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False).alias(col)
        )

if "date_reso_ord" in city_properties.columns and city_properties["date_reso_ord"].dtype == pl.String:
    city_properties = city_properties.with_columns(
        pl.col("date_reso_ord").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
    )

print("Date columns parsed.")
print("Business certs dtypes:", {c: str(business_certs[c].dtype) for c in business_date_cols})

Date columns parsed.
Business certs dtypes: {'date_account_creation': "Datetime(time_unit='us', time_zone=None)", 'date_cert_expiration': "Datetime(time_unit='us', time_zone=None)", 'date_cert_effective': "Datetime(time_unit='us', time_zone=None)", 'date_business_start': "Datetime(time_unit='us', time_zone=None)"}


## 5. Analysis

### 5a. Business Tax Certificates: Property Management and Real Estate

In [22]:
rental_keywords = [
    "property management",
    "real estate",
    "rental",
    "landlord",
    "apartment",
    "lessor",
]

pattern = "(?i)" + "|".join(rental_keywords)

rental_businesses = business_certs.filter(
    pl.col("naics_description").str.contains(pattern)
    | pl.col("dba_name").str.contains(pattern)
    | pl.col("business_owner_name").str.contains(pattern)
)

print(f"Rental/property-related businesses: {rental_businesses.shape[0]:,}")
print(f"\nNAICS descriptions found:")
rental_businesses["naics_description"].drop_nulls().unique().sort().head(20)

Rental/property-related businesses: 28,248

NAICS descriptions found:


naics_description
str
"""ACCOMMODATION"""
"""ACCOUNTING/TAX PREP/BOOKKEEP/P…"
"""ACTIVITIES RELATED TO REAL EST…"
"""ADMIN URBAN PLANNING & COMMUNI…"
"""ADMINISTRATIVE & SUPPORT SERVI…"
…
"""ALL OTHER CONSUMER GOODS RENTA…"
"""ALL OTHER FINANCIAL INVESTMENT…"
"""ALL OTHER GENERAL MERCHANDISE …"


In [23]:
top_rental = (
    rental_businesses
    .group_by("business_owner_name")
    .agg(
        pl.col("address_road").n_unique().alias("address_count"),
        pl.col("dba_name").first(),
        pl.col("account_status").first(),
    )
    .sort("address_count", descending=True)
    .head(25)
)
print("Top rental/property businesses by number of unique addresses:")
top_rental

Top rental/property businesses by number of unique addresses:


business_owner_name,address_count,dba_name,account_status
str,u32,str,str
"""REDBOX AUTOMATED RETAIL LLC""",50,"""REDBOX AUTOMATED RETAIL LLC""","""Cancelled"""
"""UNKNOWN""",38,"""SHITHAM REAL ESTATE""","""Cancelled"""
"""HOMESERVICES OF CALIFORNIA INC""",9,"""BERKSHIRE HATHAWAY HOMESERVICE…","""Active"""
"""PICKFORD REALTY LTD""",8,"""BERKSHIRE HATHAWAY HOMESERVICE…","""Active"""
"""BLOCKBUSTER LLC""",8,"""BLOCKBUSTER LLC""","""Cancelled"""
…,…,…,…
"""STEPHANIE GABRIEL""",4,"""STEPHANIE GABRIEL""","""Cancelled"""
"""CUSHFIELD MAINTENANCE WEST COR…",4,"""CUSHMAN & WAKEFIELD""","""Cancelled"""
"""COAST LEASING CORP""",4,"""ADVANTAGE RENT-A-CAR""","""Cancelled"""


### 5b. STRO Licenses

In [24]:
from datetime import datetime

stro_active = stro.filter(
    pl.col("date_expiration") >= datetime.now()
)
print(f"Total STRO licenses: {stro.shape[0]:,}")
print(f"Active (not expired) STRO licenses: {stro_active.shape[0]:,}")

Total STRO licenses: 8,328
Active (not expired) STRO licenses: 0


In [25]:
tier_counts = (
    stro
    .group_by("tier")
    .len()
    .sort("len", descending=True)
)
print("STRO licenses by tier:")
tier_counts

STRO licenses by tier:


tier,len
str,u32
"""Tier 3""",4715
"""Tier 2""",2369
"""Tier 4""",1097
"""Tier 1""",147


In [26]:
community_density = (
    stro
    .group_by("community_planning_area")
    .len()
    .sort("len", descending=True)
)
print("STRO density by community planning area:")
community_density.head(20)

STRO density by community planning area:


community_planning_area,len
str,u32
"""MISSION BEACH""",1315
"""PACIFIC BEACH""",1181
"""LA JOLLA""",787
"""UPTOWN""",588
"""OCEAN BEACH""",564
…,…
"""MIRA MESA""",112
"""MID-CITY:EASTERN AREA""",112
"""SERRA MESA""",95


In [27]:
stro_with_coords = stro.filter(
    pl.col("latitude").is_not_null() & pl.col("longitude").is_not_null()
)

m_stro = folium.Map(location=[32.7157, -117.1611], zoom_start=11)

heat_data = [
    [row["latitude"], row["longitude"]]
    for row in stro_with_coords.select("latitude", "longitude").to_dicts()
]
HeatMap(heat_data, radius=10, blur=15).add_to(m_stro)

print(f"Mapping {len(heat_data):,} STRO locations")
m_stro

Mapping 8,326 STRO locations


### 5c. TOT Establishments

In [28]:
print("=== TOT status values ===")
print(tot["status"].value_counts().sort("count", descending=True))
print("\n=== TOT certificate_type values ===")
print(tot["certificate_type"].value_counts().sort("count", descending=True))

tot_by_zip = (
    tot
    .group_by("property_zip")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print("\nTOT establishment concentration by zip code:")
tot_by_zip.head(20)

=== TOT status values ===
shape: (1, 2)
┌────────┬───────┐
│ status ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ 1.0    ┆ 15050 │
└────────┴───────┘

=== TOT certificate_type values ===
shape: (1, 2)
┌──────────────────┬───────┐
│ certificate_type ┆ count │
│ ---              ┆ ---   │
│ f64              ┆ u32   │
╞══════════════════╪═══════╡
│ 7.0              ┆ 15050 │
└──────────────────┴───────┘

TOT establishment concentration by zip code:


property_zip,count
i64,u32
92109,3697
92037,1414
92101,1362
92107,1081
92104,952
…,…
92126,208
92122,204
92114,201


In [29]:
print("TOT certificate types:")
tot.group_by("certificate_type").len().sort("len", descending=True)

TOT certificate types:


certificate_type,len
f64,u32
7.0,15050


### 5d. City Leases

In [30]:
active_leases = city_leases.filter(pl.col("lease_status") == "Active")
print(f"Active city leases: {active_leases.shape[0]:,}")
print(f"\nTop active leases:")
active_leases.select(
    "lessee_name", "lessee_company", "lease_description",
    "cost_line_amt_USD", "date_effective", "date_sched_termination"
).head(20)

Active city leases: 0

Top active leases:


lessee_name,lessee_company,lease_description,cost_line_amt_USD,date_effective,date_sched_termination
str,str,str,str,datetime[μs],datetime[μs]


In [31]:
top_lessees = (
    active_leases
    .filter(pl.col("lessee_company").is_not_null())
    .group_by("lessee_company")
    .agg(pl.len().alias("lease_count"))
    .sort("lease_count", descending=True)
)
print("Top lessees by lease count:")
top_lessees.head(20)

Top lessees by lease count:


lessee_company,lease_count
str,u32


## 6. Landlord Profile Join

Cross-reference building permit holders with business tax certificate registrants to identify confirmed landlords.

In [32]:
PERMIT_OVERRIDES = {
    "JOB_APN": pl.String,
    "JOB_BC_CODE": pl.String,
    "APPROVAL_FLOOR_AREA": pl.Float64,
    "APPROVAL_STORIES": pl.Float64,
    "APPROVAL_VALUATION": pl.Float64,
}

permit_files = [
    "permits_set1_active_datasd.csv",
    "permits_set1_closed_datasd.csv",
    "permits_set2_active_datasd.csv",
    "permits_set2_closed_datasd.csv",
]

permit_dfs = []
for f in permit_files:
    path = PERMITS_DIR / f
    if path.exists():
        df = pl.read_csv(path, infer_schema_length=10000, schema_overrides=PERMIT_OVERRIDES)
        df = df.rename({c: c.upper() for c in df.columns})
        permit_dfs.append(df)
        print(f"Loaded {f}: {df.shape[0]:,} rows")

permits = pl.concat(permit_dfs, how="diagonal_relaxed")
permits = empty_to_null(permits)

# Cast lat/lng to Float64
permits = permits.with_columns(
    pl.col("LAT_JOB").cast(pl.Float64, strict=False),
    pl.col("LNG_JOB").cast(pl.Float64, strict=False),
)
print(f"\nTotal permits: {permits.shape[0]:,} rows")
print(f"LAT_JOB dtype: {permits['LAT_JOB'].dtype}")

Loaded permits_set1_active_datasd.csv: 263,306 rows
Loaded permits_set1_closed_datasd.csv: 548,631 rows
Loaded permits_set2_active_datasd.csv: 259,847 rows
Loaded permits_set2_closed_datasd.csv: 132,550 rows

Total permits: 1,204,334 rows
LAT_JOB dtype: Float64


In [33]:
permit_holders = (
    permits
    .filter(pl.col("APPROVAL_PERMIT_HOLDER").is_not_null())
    .select(
        pl.col("APPROVAL_PERMIT_HOLDER").str.to_uppercase().str.strip_chars().alias("name"),
        pl.col("ADDRESS_JOB").alias("permit_address"),
    )
)

permit_names = (
    permit_holders
    .group_by("name")
    .agg(
        pl.len().alias("permit_count"),
        pl.col("permit_address").n_unique().alias("permit_addresses"),
    )
)

business_names = (
    business_certs
    .filter(pl.col("business_owner_name").is_not_null())
    .with_columns(
        pl.col("business_owner_name").str.to_uppercase().str.strip_chars().alias("name")
    )
    .group_by("name")
    .agg(
        pl.len().alias("business_cert_count"),
        pl.col("address_road").n_unique().alias("business_addresses"),
    )
)

landlord_profiles = (
    permit_names
    .join(business_names, on="name", how="inner")
    .with_columns(
        (pl.col("permit_addresses") + pl.col("business_addresses")).alias("total_addresses")
    )
    .sort("total_addresses", descending=True)
)

print(f"Confirmed landlord profiles (name in both permits and business certs): {landlord_profiles.shape[0]:,}")
landlord_profiles.head(25)

Confirmed landlord profiles (name in both permits and business certs): 8,211


name,permit_count,permit_addresses,business_cert_count,business_addresses,total_addresses
str,u32,u32,u32,u32,u32
"""SOLAR TECH ENERGY SYSTEMS INC""",1497,1417,2,1,1418
"""BILL HOWE HEATING & AIR CONDIT…",966,912,1,1,913
"""INFINITY ENERGY INC""",914,867,1,1,868
"""WEST COAST APPLIANCE SERVICES …",759,720,1,1,721
"""M N MAUZY MECHANICAL INC""",579,553,1,1,554
…,…,…,…,…,…
"""ALL AMERICAN PLUMBING INC""",257,247,1,1,248
"""DAN ANDERSON""",239,225,1,1,226
"""NOVA SECURITY""",428,225,1,1,226


## 7. Landlord Lookup Function

In [34]:
def landlord_lookup(name: str) -> dict:
    """Searches business tax certs and permit holders for a name.

    Returns all associated addresses and activity for the given name.
    """
    name_upper = name.strip().upper()

    biz_matches = business_certs.filter(
        pl.col("business_owner_name").str.to_uppercase().str.contains(name_upper)
        | pl.col("dba_name").str.to_uppercase().str.contains(name_upper)
    )

    permit_matches = permits.filter(
        pl.col("APPROVAL_PERMIT_HOLDER").str.to_uppercase().str.contains(name_upper)
    )

    biz_addresses = (
        biz_matches
        .select("address_no", "address_road", "address_sfx", "address_city",
                "address_zip", "account_status", "dba_name", "naics_description")
        .unique()
    )

    permit_addresses = (
        permit_matches
        .select("ADDRESS_JOB", "APPROVAL_TYPE", "APPROVAL_STATUS", "PROJECT_TITLE")
        .unique()
    )

    result = {
        "name_searched": name,
        "business_cert_count": biz_matches.shape[0],
        "permit_count": permit_matches.shape[0],
        "business_addresses": biz_addresses,
        "permit_addresses": permit_addresses,
    }

    print(f"Landlord lookup: '{name}'")
    print(f"  Business certificates: {biz_matches.shape[0]}")
    print(f"  Building permits: {permit_matches.shape[0]}")

    if biz_addresses.shape[0] > 0:
        print(f"\n  Business addresses ({biz_addresses.shape[0]}):")
        print(biz_addresses.head(15))

    if permit_addresses.shape[0] > 0:
        print(f"\n  Permit addresses ({permit_addresses.shape[0]}):")
        print(permit_addresses.head(15))

    return result

## 8. Address Rental Profile Function

In [35]:
def address_rental_profile(
    lat: float,
    lng: float,
    street_number: str,
    street_name: str,
    radius_miles: float = 0.5,
) -> dict:
    """Builds a rental profile for a location.

    Args:
        lat: Latitude.
        lng: Longitude.
        street_number: Street number (e.g. "1810").
        street_name: Street name without suffix (e.g. "STATE").
        radius_miles: Radius for STRO density count.
    """
    st_num = street_number.strip()
    st_name = street_name.strip().upper()

    # STRO density within radius (uses coords)
    deg_per_mile = 1.0 / 69.0
    lat_delta = radius_miles * deg_per_mile
    lng_delta = radius_miles * deg_per_mile / max(math.cos(math.radians(lat)), 0.01)

    nearby_stro = stro.filter(
        pl.col("latitude").is_not_null()
        & pl.col("longitude").is_not_null()
        & (pl.col("latitude") >= lat - lat_delta)
        & (pl.col("latitude") <= lat + lat_delta)
        & (pl.col("longitude") >= lng - lng_delta)
        & (pl.col("longitude") <= lng + lng_delta)
    )

    # Is this exact address a licensed STR?
    is_stro = stro.filter(
        (pl.col("street_number").cast(pl.String).str.strip_chars() == st_num)
        & pl.col("street_name").str.to_uppercase().str.contains(st_name)
    ).shape[0] > 0

    # Business tax certs: exact address match only
    biz_at_address = business_certs.filter(
        pl.col("address_no").is_not_null()
        & pl.col("address_road").is_not_null()
        & (pl.col("address_no").str.strip_chars() == st_num)
        & pl.col("address_road").str.to_uppercase().str.contains(st_name)
    )

    # Permits at this address
    addr_pattern = st_num + ".*" + st_name
    permit_at_address = permits.filter(
        pl.col("ADDRESS_JOB").is_not_null()
        & pl.col("ADDRESS_JOB").str.to_uppercase().str.contains(addr_pattern)
    )

    # Most recent permit and who has done work
    last_permit_date = None
    permit_holders = []
    if permit_at_address.shape[0] > 0:
        dated = permit_at_address.filter(pl.col("DATE_APPROVAL_CREATE").is_not_null())
        if dated.shape[0] > 0:
            sorted_permits = dated.sort("DATE_APPROVAL_CREATE", descending=True)
            last_permit_date = sorted_permits["DATE_APPROVAL_CREATE"][0]
        permit_holders = (
            permit_at_address["APPROVAL_PERMIT_HOLDER"]
            .drop_nulls()
            .unique()
            .to_list()
        )

    profile = {
        "street_number": st_num,
        "street_name": st_name,
        "is_known_stro": is_stro,
        "stro_count_in_radius": nearby_stro.shape[0],
        "nearby_stro_df": nearby_stro,
        "business_certs_at_address": biz_at_address.shape[0],
        "business_certs_df": biz_at_address,
        "permits_at_address": permit_at_address.shape[0],
        "last_permit_date": last_permit_date,
        "permit_holders": permit_holders,
    }

    print(f"Address Rental Profile: {st_num} {st_name}")
    print(f"  Known STRO at address: {is_stro}")
    print(f"  Licensed STRs within {radius_miles} mi: {nearby_stro.shape[0]}")
    print()
    print(f"  Business tax certs at address: {biz_at_address.shape[0]}")
    if biz_at_address.shape[0] > 0:
        print(biz_at_address.select(
            "business_owner_name", "dba_name", "naics_description", "account_status"
        ))
    print()
    print(f"  Permits at address: {permit_at_address.shape[0]}")
    print(f"  Last permit activity: {last_permit_date}")
    if permit_holders:
        print(f"  Who has done work on this building ({len(permit_holders)} entities):")
        for h in permit_holders[:15]:
            print(f"    {h}")
    print()

    return profile

## 9. Query: 1810 State St

In [36]:
profile_1810 = address_rental_profile(32.7240, -117.1669, "1810", "STATE")

Address Rental Profile: 1810 STATE
  Known STRO at address: False
  Licensed STRs within 0.5 mi: 263

  Business tax certs at address: 10
shape: (10, 4)
┌───────────────────────────┬──────────────────────────┬──────────────────────────┬────────────────┐
│ business_owner_name       ┆ dba_name                 ┆ naics_description        ┆ account_status │
│ ---                       ┆ ---                      ┆ ---                      ┆ ---            │
│ str                       ┆ str                      ┆ str                      ┆ str            │
╞═══════════════════════════╪══════════════════════════╪══════════════════════════╪════════════════╡
│ GREYSTAR REAL ESTATE      ┆ EIGHTEEN TEN STATE       ┆ REAL ESTATE PROPERTY     ┆ Active         │
│ PARTNERS …                ┆                          ┆ MANAGERS                 ┆                │
│ SAVVY APPS LLC            ┆ SAVVY APPS LLC           ┆ CUSTOM COMPUTER          ┆ Active         │
│                           ┆          

In [37]:
m_1810 = folium.Map(location=[32.7240, -117.1669], zoom_start=15)

folium.Marker(
    [32.7240, -117.1669],
    popup="1810 State St",
    icon=folium.Icon(color="red", icon="home"),
).add_to(m_1810)

nearby = profile_1810["nearby_stro_df"]
for row in nearby.to_dicts():
    if row.get("latitude") and row.get("longitude"):
        folium.CircleMarker(
            [row["latitude"], row["longitude"]],
            radius=4,
            color="blue",
            fill=True,
            fill_opacity=0.6,
            popup=f"STRO: {row.get('address', 'N/A')}",
        ).add_to(m_1810)

folium.Circle(
    [32.7240, -117.1669],
    radius=804.672,
    color="red",
    fill=False,
    weight=2,
    popup="0.5 mile radius",
).add_to(m_1810)

print(f"STRO density map: {nearby.shape[0]} licensed STRs within 0.5 miles")
m_1810

STRO density map: 263 licensed STRs within 0.5 miles
